# Week 1: Can a 2-4 team still save its season?

**Series:** Save the Season (NFL) · **Concept:** luck vs skill

Set `TEAM` below, then run **Restart Kernel and Run All Cells**, or from the repo root:

```powershell
analysis/.venv/Scripts/python analysis/tools/run_notebook.py analysis/week01-save-the-season.ipynb --param TEAM=PHI
```

Before committing: clear all outputs and run `analysis/tools/check_notebooks.py`.

In [ ]:
TEAM = "DAL"
SEASON = None  # None means the latest season with completed games
THROUGH_WEEK = None  # None means the latest week whose games are all final

In [ ]:
import sys
from dataclasses import asdict
from pathlib import Path

# Find analysis/ whether this notebook runs there, in a subfolder, or from the repo root.
here = Path.cwd().resolve()
ANALYSIS = next(d for d in (here, *here.parents, here / "analysis") if (d / "lib" / "style.py").is_file())
sys.path.insert(0, str(ANALYSIS))

import pandas as pd
from matplotlib.ticker import PercentFormatter

from lib import nfl_queries, nflverse, style, team_posts
from lib import nfl_team_report as report
from lib.db import read_engine

engine = read_engine()
style.apply_style()

## 1. The question

The team is 2-4 (or close to it). Is the season already gone, or can they still make the playoffs?

## 2. The concept: luck vs skill

A team's record tells you what happened, but a few close games can bend it a long way. Points scored and allowed
tell you more about how well a team has actually played. **Pythagorean wins** turn those points into the record a
team "should" have. Then we ask a second question: how much of a team's early form usually lasts? We measure that
from past seasons, and use it to play out the rest of this season thousands of times.

## 3. Data

- **This season:** regular-season scores, per-play efficiency and turnovers from the Lineform database (read only),
  through the last week whose games are all final.
- **History:** regular-season scores from 2012 on, to measure how much early point differential lasts, the home edge
  and how much single games swing.
- **Playoff bar:** nflverse schedules for the last five completed seasons (17 games, 14 playoff teams), because the
  database stores regular-season games only. Cached in `analysis/data/` after the first download.

In [ ]:
history = nfl_queries.games(engine, 2012, 2100)
season = SEASON or int(history.loc[history["status"] == "completed", "season"].max())
through = THROUGH_WEEK or report.last_final_week(history, season)
recent_seasons = list(range(season - 5, season))      # 17-game, 14-team playoff era for 2026
slope_seasons = list(range(2012, season))
carryover_seasons = list(range(2024, season))         # player turnovers are stored from 2024

efficiency = nfl_queries.efficiency(engine, season, season)
giveaways = nfl_queries.giveaways(engine, min(carryover_seasons), season)
schedules = nflverse.load_schedules(recent_seasons)

this_season = report.played_between(history, season, 1, through)
print(f"{TEAM} {season}, data through week {through}")
print(f"This season: {this_season['game_id'].nunique()} games played; history: {len(slope_seasons)} seasons "
      f"({slope_seasons[0]}-{slope_seasons[-1]}); playoff bar: {recent_seasons[0]}-{recent_seasons[-1]} "
      f"({schedules['game_type'].isin(['WC', 'DIV']).sum()} playoff games)")

## 4. Method

1. **Record and luck:** compare actual wins with Pythagorean wins (points scored to the 2.37 power, over that plus
   points allowed to the same power). Count one-score games (8 points or fewer) and turnover margin.
2. **How good are they per play?** Rank offense and defense by EPA per play (*expected points added: how much each
   play helps the score*) and success rate.
3. **Rating:** point differential per game, shrunk toward zero by the share that has carried into the rest of the
   season since 2012, measured at this same week.
4. **Rest of the season:** each remaining game's win chance comes from the two ratings plus the home edge, with the
   usual spread of game margins. Play the schedule 10,000 times.
5. **Playoff bar:** in recent seasons, how often did teams with each final win total make the playoffs? Combine that
   with the simulated final wins.

This is a **rough estimate**: no division races or tiebreakers, international games count as home games, and the
rules were fixed before looking at any team.

## 5. Analysis

In [ ]:
fit = report.fit_league(history, slope_seasons=slope_seasons, margin_seasons=recent_seasons, through_week=through)
print(f"Early point differential that lasts: {fit.slope:.0%} ({fit.slope_team_seasons} team-seasons)")
print(f"Home edge: {fit.home_edge:.1f} points; typical swing of a game margin: {fit.sigma:.1f} points")

carryover = report.turnover_carryover(giveaways, history, carryover_seasons, through)
print(f"Turnover margin through week {through} vs the rest of the season: "
      f"r = {carryover.correlation:.2f} ({carryover.team_seasons} team-seasons, a small sample)")

In [ ]:
playoff_table = report.playoff_rate_by_wins(schedules)
bar = report.playoff_bar(playoff_table)
print(f"Playoff bar: {bar.made} of {bar.teams} teams with {bar.wins} wins made it")
playoff_table

In [ ]:
result = report.team_report(TEAM, season, through, games=history, efficiency=efficiency, giveaways=giveaways,
                            playoff_table=playoff_table, fit=fit)
snapshot = {k: v for k, v in asdict(result).items() if k != "final_win_distribution"}
pd.Series(snapshot, name=report.team_name(TEAM)).to_frame()

In [ ]:
league = report.standings(history, season, through)
league = league.assign(luck_gap=league["pythagorean_wins"] - (league["wins"] + league["ties"] / 2))
league.sort_values("luck_gap", ascending=False)[["wins", "losses", "ties", "point_diff", "pythagorean_wins",
                                                  "luck_gap"]].round(2).head(10)

## 6. The answer

Printed below from the numbers, then the main caveat.

In [ ]:
print(f"The {report.team_name(TEAM)} are {team_posts.record(result)}, and {team_posts.season_sentence(result)}.")
print(team_posts.luck_sentence(result))
print(f"Caveat: {through} weeks is a small sample, and this rough estimate has no tiebreakers or division races.")

## 7. Chart export

One point: the playoff chance. Bars are the share of simulated seasons ending at each win total; win totals at or
above the playoff bar are logo blue.

In [ ]:
CHART_NAME = f"week01-{TEAM.lower()}"
nickname = report.nickname(TEAM)
distribution = result.final_win_distribution
totals = list(range(min(distribution), max(distribution) + 1))
shares = [distribution.get(w, 0.0) for w in totals]
title = f"The {nickname} make the playoffs in {team_posts.chance_phrase(result.playoff_chance)} seasons"
subtitle = f"{team_posts.record(result)} through week {through} · 10,000 simulated seasons"
# The blue bars alone undercount the chance: teams one win short sometimes got in too.
at_or_above = playoff_table.loc[playoff_table.index >= bar.wins, ["made", "teams"]].sum()
note = f"{bar.wins}+ wins:\n{at_or_above['made']} of {at_or_above['teams']}\nmade it"
if bar.wins - 1 in playoff_table.index:
    below = playoff_table.loc[bar.wins - 1]
    note += f"\n\n{bar.wins - 1} wins:\n{int(below['made'])} of {int(below['teams'])}"
source = f"Source: Lineform database, nflverse · data through week {through}, {season}"

for size in style.SIZES:
    fig, ax = style.new_figure(size)
    ax.bar(totals, shares, color=[style.ACCENT if w >= bar.wins else style.GRAY for w in totals], width=0.8)
    ax.axvline(bar.wins - 0.5, color=style.TEXT_2, linewidth=2, linestyle="--")
    ax.text(bar.wins - 0.35, max(shares) * 1.02, note,
            color=style.TEXT, va="top", fontproperties=style.font("body_bold"), fontsize=style.px(30))
    ax.set_xticks(totals)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_xlabel("Final wins")
    ax.set_ylabel("Share of simulated seasons")
    style.finish(fig, ax, title=title, subtitle=subtitle, source=source)
    print(style.export(fig, CHART_NAME, size))

### Thumbnail

Headline: 3 to 5 words. No team logos or player photos. The config is saved in `thumbnails/configs/` so the
thumbnail can be rebuilt later.

In [ ]:
import json

from thumbnails.config import ANALYSIS_DIR, load_config
from thumbnails.make_thumbnail import render

thumbnail = {
    "headline": f"Can the {nickname} recover?",
    "subline": f"{team_posts.record(result)} through week {through}",
    "chart": f"{CHART_NAME}-feed.png",
    "sport": "nfl",
    "callout": {"value": team_posts.callout_value(result.playoff_chance), "label": "chance to make the playoffs"},
    "name": CHART_NAME,
}
config_path = ANALYSIS_DIR / "thumbnails" / "configs" / f"{CHART_NAME}.json"
config_path.write_text(json.dumps(thumbnail, indent=2) + "\n", encoding="utf-8")
render(load_config(config_path))

## 8. Draft posts

Generated from the numbers above into `output/week01-<team>-posts.md`. Edit them before posting. No betting advice,
no picks, no claims the data does not support.

In [ ]:
posts = team_posts.build_posts(result, bar=bar, turnover_carryover=carryover.correlation)
posts_path = style.OUTPUT_DIR / f"{CHART_NAME}-posts.md"
posts_path.write_text(team_posts.to_markdown(result, posts), encoding="utf-8")
print(posts_path)
print(team_posts.to_markdown(result, posts))

## 9. What I learned

- **About the concept:** a record and how well a team has played are different things; close games and turnovers
  can open a gap between them.
- **About the method:** early-season form only partly lasts, so a fair rating keeps a little under half of it, and a
  playoff answer is a range of outcomes, not a yes or no.
- **Something that surprised me:** check the follow-up clip draft; it names the most surprising number for this team.

**New glossary terms** (added to `lib/glossary.md`): point differential, Pythagorean wins, one-score game, turnover
margin, EPA per play, success rate, regression to the mean, strength of schedule, simulation.